In [178]:
import pandas as pd

In [179]:
df = pd.read_csv('/kaggle/input/datasets/prabhavsinghal/my-dataset/100_Unique_QA_Dataset.csv')

In [180]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [181]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")
    return text.split()

In [182]:
tokenize('What is the capital of INdia?')

['what', 'is', 'the', 'capital', 'of', 'india']

In [183]:
# vocab
vocab = {'<UNK>':0}

In [184]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens = tokenized_question + tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

  

In [185]:
df.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [186]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [187]:
len(vocab)

324

In [188]:
# convert words to numerical indeces
def text_to_indeces(text,vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text
        

In [189]:
text_to_indeces('What is the first element in periodic table',vocab)

[1, 2, 3, 141, 117, 19, 277, 278]

In [190]:
import torch
from torch.utils.data import Dataset,DataLoader

In [191]:
class QADataset(Dataset):
    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
        
    def __getitem__(self,index):
        numerical_question = text_to_indeces(self.df.iloc[index]['question'],self.vocab)
        numerical_answer = text_to_indeces(self.df.iloc[index]['answer'],self.vocab)
        return torch.tensor(numerical_question) , torch.tensor(numerical_answer)



In [192]:
dataset = QADataset(df,vocab)

In [193]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [194]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [195]:
for question,answer in dataloader:
    print(question,answer)

tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[ 78,  79, 150, 151,  14, 152, 153]]) tensor([[154]])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([[285]])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([[36]])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([[156]])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([[134]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([[124]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([[61]])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([[170]])
tensor([[ 10,  75, 111]]) tensor([[112]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[ 10,  11, 157, 158, 159]]) tensor([[160]])
tensor([[ 42, 137, 118,   3, 247,   5, 2

In [196]:
import torch.nn as nn

In [197]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.RNN(50,64,batch_first=True)
        self.fc = nn.Linear(64,vocab_size)


    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden,final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output
        

In [198]:
learning_rate = 0.001
epochs = 100

In [199]:
model = SimpleRNN(len(vocab))

In [200]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = learning_rate)

In [201]:
# training
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        output = model(question)
        loss = criterion(output,answer[0])
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()

    print(f"epoch: {epoch+1} , loss: {total_loss:4f}")

epoch: 1 , loss: 524.623028
epoch: 2 , loss: 451.322003
epoch: 3 , loss: 369.864954
epoch: 4 , loss: 312.996921
epoch: 5 , loss: 261.919119
epoch: 6 , loss: 214.908134
epoch: 7 , loss: 172.283660
epoch: 8 , loss: 133.881044
epoch: 9 , loss: 103.819392
epoch: 10 , loss: 79.389740
epoch: 11 , loss: 61.226649
epoch: 12 , loss: 48.135967
epoch: 13 , loss: 38.448432
epoch: 14 , loss: 31.177736
epoch: 15 , loss: 25.741698
epoch: 16 , loss: 21.553431
epoch: 17 , loss: 18.212269
epoch: 18 , loss: 15.634865
epoch: 19 , loss: 13.481143
epoch: 20 , loss: 11.773041
epoch: 21 , loss: 10.148212
epoch: 22 , loss: 8.959794
epoch: 23 , loss: 7.969494
epoch: 24 , loss: 7.072221
epoch: 25 , loss: 6.360561
epoch: 26 , loss: 5.697144
epoch: 27 , loss: 5.186140
epoch: 28 , loss: 4.683985
epoch: 29 , loss: 4.276685
epoch: 30 , loss: 3.904620
epoch: 31 , loss: 3.577315
epoch: 32 , loss: 3.294137
epoch: 33 , loss: 3.042803
epoch: 34 , loss: 2.817511
epoch: 35 , loss: 2.611399
epoch: 36 , loss: 2.417126
epoch: 

In [211]:
def predict(model,question,threshold=0.3):
    numerical_question = text_to_indeces(question,vocab)
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    output = model(question_tensor)
    probs = torch.nn.functional.softmax(output,dim=1)
    value,index = torch.max(probs ,dim=1)
    if value<threshold :
        print("I don't Know")
    else:    
        print(list(vocab.keys())[index])
        
    

In [215]:
predict(model , 'What is the animated TV show?')

simpsons
